In [21]:
import os
import numpy as np
import pandas as pd
import logging
from scipy.stats import ttest_ind

from sqlalchemy import create_engine, text

In [2]:
def download():

    from urllib.request import urlretrieve
    import os

    url = (
        "https://raw.githubusercontent.com/Explore-AI/Public-Data/master/"
        "Maji_Ndogo/Maji_Ndogo_farm_survey_small.db"
    )

    db_file = "Maji_Ndogo_farm_survey_small.db"

    # Download only if the database is not already present
    if not os.path.exists(db_file):
        urlretrieve(url, db_file)
        print(f"Downloaded '{db_file}'.")
    else:
        print(f"'{db_file}' already exists.")

    return


In [3]:
from data_ingestion import create_db_engine, query_data, read_from_web_CSV

print(create_db_engine.__module__)
print(query_data.__module__)
print(read_from_web_CSV.__module__) 

data_ingestion
data_ingestion
data_ingestion


In [4]:
print(create_db_engine.__name__)
print(query_data.__name__)
print(read_from_web_CSV.__name__)


create_db_engine
query_data
read_from_web_CSV


In [28]:
config_params = {
    "sql_query": """
    SELECT *
    FROM geographic_features
    LEFT JOIN weather_features USING (Field_ID)
    LEFT JOIN soil_and_crop_features USING (Field_ID)
    LEFT JOIN farm_management_features USING (Field_ID)
    """,

    "db_path": "sqlite:///Maji_Ndogo_farm_survey_small.db",

    "columns_to_rename": {
        "Annual_yield": "Crop_type",
        "Crop_type": "Annual_yield",
    },

    "values_to_rename": {
        "cassaval": "cassava",
        "wheatn": "wheat",
        "teaa": "tea",
    },

    "weather_csv_path": "Weather_station_data.csv",

    "weather_mapping_csv": "Weather_data_field_mapping.csv",
}

In [29]:
from field_data_processor import FieldDataProcessor

In [30]:
field_processor = FieldDataProcessor(config_params)
field_df = field_processor.process()

2026-08-07 15:50:59,710 - data_ingestion - INFO - Database engine created successfully.
2026-08-07 15:50:59,825 - data_ingestion - INFO - Query executed successfully.


In [31]:
print("Shape:", field_df.shape)

Shape: (5654, 19)


In [32]:
from weather_data_processor import WeatherDataProcessor

In [33]:
weather_processor = WeatherDataProcessor(config_params)
weather_df = weather_processor.process()
print(weather_df.shape)

print(weather_df["Measurement"].value_counts())

2026-08-07 15:51:15,473 - data_ingestion - INFO - CSV file read successfully from the web.


(1843, 4)
Measurement
Temperature        611
Pollution_level    607
Rainfall           477
Name: count, dtype: int64


In [34]:
def filter_field_data(df, station_id, measurement):
    """
    Filter field data for a station and measurement.

    Parameters
    ----------
    df : pandas.DataFrame
        Processed field DataFrame.
    station_id : int
        Weather station ID.
    measurement : str
        Measurement name.

    Returns
    -------
    pandas.Series
        Filtered measurement values.
    """

    column_mapping = {
        "Temperature": "Ave_temps",
        "Rainfall": "Rainfall",
        "Pollution_level": "Pollution_level"
    }

    column = column_mapping[measurement]

    return (
        df.loc[
            df["Weather_station"] == station_id,
            column
        ]
        .dropna()
    )


def filter_weather_data(df, station_id, measurement):
    """
    Filter weather sensor data for a station and measurement.

    Parameters
    ----------
    df : pandas.DataFrame
        Weather DataFrame.
    station_id : int
        Weather station ID.
    measurement : str
        Measurement name.

    Returns
    -------
    pandas.Series
        Sensor measurement values.
    """

    return (
        df.loc[
            (df["Weather_station_ID"] == station_id) &
            (df["Measurement"] == measurement),
            "Value"
        ]
        .dropna()
    )


def run_ttest(column_a, column_b):
    """
    Run Welch's independent two-sample t-test.

    Parameters
    ----------
    column_a : array-like
        First sample.
    column_b : array-like
        Second sample.

    Returns
    -------
    tuple
        Test statistic and p-value.
    """

    t_stat, p_val = ttest_ind(
        column_a,
        column_b,
        equal_var=False,
        alternative="two-sided"
    )

    return t_stat, p_val


def print_ttest_results(station_id, measurement, p_val, alpha):
    """
    Print hypothesis test results.

    Parameters
    ----------
    station_id : int
    Weather station ID.
    measurement : str
    Measurement tested.
    p_val : float
    Test p-value.
    alpha : float
    Significance level.
        """

    if p_val > alpha:
        print(
            f"No significant difference in {measurement} at Station {station_id} "
            f"(P-value: {p_val:.5f} > {alpha}). H₀ not rejected."
        )

    else:
        print(
            f"Significant difference in {measurement} at Station {station_id} "
            f"(P-value: {p_val:.5f} ≤ {alpha}). H₀ rejected."
        )


    return (
        filter_field_data,
        filter_weather_data,
        print_ttest_results,
        run_ttest,
    )

In [35]:
measurements = [
    "Temperature",
    "Rainfall",
    "Pollution_level"
]

for station in range(5):

    for measurement in measurements:

        field_values = filter_field_data(
            field_df,
            station,
            measurement
        )

        weather_values = filter_weather_data(
            weather_df,
            station,
            measurement
        )

        t_stat, p_val = run_ttest(
            field_values,
            weather_values
        )

        print_ttest_results(
            station,
            measurement,
            p_val,
            0.05
        )
       

No significant difference in Temperature at Station 0 (P-value: 0.90761 > 0.05). H₀ not rejected.
No significant difference in Rainfall at Station 0 (P-value: 0.11056 > 0.05). H₀ not rejected.
No significant difference in Pollution_level at Station 0 (P-value: 0.56418 > 0.05). H₀ not rejected.
No significant difference in Temperature at Station 1 (P-value: 0.47241 > 0.05). H₀ not rejected.
No significant difference in Rainfall at Station 1 (P-value: 0.45007 > 0.05). H₀ not rejected.
No significant difference in Pollution_level at Station 1 (P-value: 0.24410 > 0.05). H₀ not rejected.
No significant difference in Temperature at Station 2 (P-value: 0.88671 > 0.05). H₀ not rejected.
No significant difference in Rainfall at Station 2 (P-value: 0.27958 > 0.05). H₀ not rejected.
No significant difference in Pollution_level at Station 2 (P-value: 0.99388 > 0.05). H₀ not rejected.
No significant difference in Temperature at Station 3 (P-value: 0.66445 > 0.05). H₀ not rejected.
No significant di